[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/054.%20The%20Production%20Order%20Quantity%20%28POQ%29%20Problem/P54-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 54. The Production Order Quantity (POQ) Problem

## Tier 1: The Pen & Paper Method

### Key assumptions
- Production rate exceeds demand rate (p > d)
- Production and demand occur continuously over time
- Setup costs are incurred each time production starts
- Holding costs are based on average inventory levels
- All parameters are known and constant

### Approach (step-by-step)
1. **Problem Definition**: We need to find the optimal production batch size Q that minimizes total annual costs
2. **Cost Components**: Total cost = Setup cost + Holding cost
3. **Mathematical Model**: Formulate the total cost function and optimize analytically
4. **Integer Solution**: Since production quantities must be integer, evaluate floor/ceil of optimal solution
5. **Validation**: Verify solution meets all constraints and requirements

### What to look for in the results
- Optimal production quantity Q* that minimizes total cost
- Maximum inventory level during production cycle
- Total annual cost breakdown (setup vs holding)
- Production cycle length and consumption cycle length

### Concrete example (from the source)
Tablet manufacturing facility:
- Annual demand: D = 60,000 tablets
- Setup cost: S = $800 per run
- Holding cost: H = $2 per tablet per year
- Production rate: p = 500 tablets/day
- Daily demand: d = 200 tablets/day (assuming 300 working days)

In [1]:
# Import required libraries for mathematical calculations and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
@dataclass
class POQParameters:
    """Data class to store POQ problem parameters"""
    D: float  # Annual demand (units/year)
    S: float  # Setup cost per production run ($/setup)
    H: float  # Annual holding cost per unit ($/unit/year)
    p: float  # Daily production rate (units/day)
    d: float  # Daily demand rate (units/day)
    working_days: int = 300  # Working days per year
    
    def __post_init__(self):
        """Validate parameters after initialization"""
        if self.p <= self.d:
            raise ValueError("Production rate must exceed demand rate (p > d)")
        if self.D <= 0 or self.S <= 0 or self.H <= 0:
            raise ValueError("Demand, setup cost, and holding cost must be positive")

# Define the concrete example from the source
params = POQParameters(
    D=60000,    # Annual demand: 60,000 tablets
    S=800,      # Setup cost: $800 per run
    H=2,        # Holding cost: $2 per tablet per year
    p=500,      # Production rate: 500 tablets/day
    d=200,      # Daily demand: 200 tablets/day
    working_days=300
)

print(f"POQ Problem Parameters:")
print(f"Annual Demand (D): {params.D:,} units")
print(f"Setup Cost (S): ${params.S:,} per run")
print(f"Holding Cost (H): ${params.H} per unit per year")
print(f"Production Rate (p): {params.p} units/day")
print(f"Daily Demand (d): {params.d} units/day")
print(f"Working Days: {params.working_days} days/year")

In [ ]:
def calculate_optimal_poq(params: POQParameters) -> Tuple[float, float, Dict]:
    """
    Calculate the optimal Production Order Quantity using analytical solution
    
    Arguments:
    params: POQParameters object containing all problem parameters
    
    Returns:
    optimal_q: Optimal production quantity (continuous)
    optimal_q_int: Optimal integer production quantity
    results: Dictionary containing detailed analysis results
    """
    
    # Calculate the optimal production quantity using the analytical formula
    # Q* = sqrt(2DS / (H * (1 - d/p)))
    demand_rate_ratio = params.d / params.p
    denominator = params.H * (1 - demand_rate_ratio)
    
    optimal_q = np.sqrt(2 * params.D * params.S / denominator)
    
    # Since we need integer solutions, evaluate floor and ceil
    q_floor = int(np.floor(optimal_q))
    q_ceil = int(np.ceil(optimal_q))
    
    # Calculate total cost for both integer options
    def total_cost(q):
        setup_cost = (params.D * params.S) / q
        holding_cost = (params.H * q / 2) * (1 - demand_rate_ratio)
        return setup_cost + holding_cost
    
    cost_floor = total_cost(q_floor)
    cost_ceil = total_cost(q_ceil)
    
    # Choose the better integer solution
    if cost_floor <= cost_ceil:
        optimal_q_int = q_floor
        optimal_cost = cost_floor
    else:
        optimal_q_int = q_ceil
        optimal_cost = cost_ceil
    
    # Calculate additional metrics for the optimal solution
    max_inventory = optimal_q_int * (1 - demand_rate_ratio)
    production_run_length = optimal_q_int / params.p
    consumption_cycle_length = optimal_q_int / params.d
    num_production_runs = params.D / optimal_q_int
    
    # Calculate cost breakdown
    setup_cost_component = (params.D * params.S) / optimal_q_int
    holding_cost_component = (params.H * optimal_q_int / 2) * (1 - demand_rate_ratio)
    
    results = {
        'optimal_q_continuous': optimal_q,
        'optimal_q_integer': optimal_q_int,
        'total_cost': optimal_cost,
        'setup_cost': setup_cost_component,
        'holding_cost': holding_cost_component,
        'max_inventory': max_inventory,
        'production_run_length_days': production_run_length,
        'consumption_cycle_length_days': consumption_cycle_length,
        'num_production_runs_per_year': num_production_runs,
        'demand_rate_ratio': demand_rate_ratio,
        'alternative_q': q_ceil if optimal_q_int == q_floor else q_floor,
        'alternative_cost': cost_ceil if optimal_q_int == q_floor else cost_floor
    }
    
    return optimal_q, optimal_q_int, results

# Calculate the optimal POQ for our example
optimal_q_cont, optimal_q_int, results = calculate_optimal_poq(params)

print(f"\n=== OPTIMAL PRODUCTION ORDER QUANTITY RESULTS ===")
print(f"Continuous Optimal Q*: {optimal_q_cont:,.2f} units")
print(f"Integer Optimal Q*: {optimal_q_int:,} units")
print(f"\nTotal Annual Cost: ${results['total_cost']:,.2f}")
print(f"  - Setup Cost: ${results['setup_cost']:,.2f}")
print(f"  - Holding Cost: ${results['holding_cost']:,.2f}")
print(f"\nMaximum Inventory Level: {results['max_inventory']:,.0f} units")
print(f"Production Run Length: {results['production_run_length_days']:.1f} days")
print(f"Consumption Cycle Length: {results['consumption_cycle_length_days']:.1f} days")
print(f"Number of Production Runs: {results['num_production_runs_per_year']:.1f} per year")

In [ ]:
# Create comprehensive visualization of POQ results
def create_poq_visualizations(params: POQParameters, results: Dict):
    """
    Create professional visualizations for POQ analysis
    
    Arguments:
    params: POQParameters object
    results: Results dictionary from calculate_optimal_poq
    """
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Production Order Quantity (POQ) Analysis Dashboard', fontsize=16, fontweight='bold')
    
    # 1. Cost Components Breakdown
    costs = [results['setup_cost'], results['holding_cost']]
    labels = ['Setup Cost', 'Holding Cost']
    colors = ['#FF6B6B', '#4ECDC4']
    
    ax1.pie(costs, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax1.set_title(f'Total Cost Breakdown\n${results["total_cost"]:,.0f}', fontweight='bold')
    
    # 2. Setup Cost Sensitivity
    setup_costs = [400, 600, 800, 1000, 1200]
    optimal_q_values = []
    
    for sc in setup_costs:
        test_params = POQParameters(params.D, sc, params.H, params.p, params.d, params.working_days)
        _, q_opt, _ = calculate_optimal_poq(test_params)
        optimal_q_values.append(q_opt)
    
    ax2.plot(setup_costs, optimal_q_values, 'o-', linewidth=2, markersize=8, color='#FF6B6B')
    ax2.set_xlabel('Setup Cost ($)', fontweight='bold')
    ax2.set_ylabel('Optimal Production Quantity', fontweight='bold')
    ax2.set_title('Setup Cost Sensitivity Analysis', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.ticklabel_format(style='plain', axis='y')
    
    # 3. Holding Cost Sensitivity
    holding_costs = [1, 1.5, 2, 2.5, 3]
    optimal_q_values_h = []
    
    for hc in holding_costs:
        test_params = POQParameters(params.D, params.S, hc, params.p, params.d, params.working_days)
        _, q_opt, _ = calculate_optimal_poq(test_params)
        optimal_q_values_h.append(q_opt)
    
    ax3.plot(holding_costs, optimal_q_values_h, 's-', linewidth=2, markersize=8, color='#4ECDC4')
    ax3.set_xlabel('Holding Cost ($/unit/year)', fontweight='bold')
    ax3.set_ylabel('Optimal Production Quantity', fontweight='bold')
    ax3.set_title('Holding Cost Sensitivity Analysis', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.ticklabel_format(style='plain', axis='y')
    
    # 4. Inventory Pattern Over Time
    # Simulate one complete production cycle
    days_production = int(np.ceil(results['production_run_length_days']))
    days_consumption = int(np.ceil(results['consumption_cycle_length_days']))
    total_days = days_consumption
    
    time_days = np.arange(0, total_days + 1)
    inventory_levels = []
    
    for day in time_days:
        if day <= days_production:
            # Production phase: inventory builds up at rate (p - d)
            inventory = day * (params.p - params.d)
        else:
            # Consumption phase: inventory depletes at rate d
            max_inventory = days_production * (params.p - params.d)
            inventory = max_inventory - (day - days_production) * params.d
        inventory_levels.append(max(0, inventory))
    
    ax4.fill_between(time_days, inventory_levels, alpha=0.3, color='#45B7D1')
    ax4.plot(time_days, inventory_levels, linewidth=3, color='#2E86AB')
    ax4.axhline(y=results['max_inventory'], color='red', linestyle='--', alpha=0.7, label=f'Max Inventory: {results["max_inventory"]:,.0f}')
    ax4.axvline(x=days_production, color='green', linestyle='--', alpha=0.7, label=f'Production End: {days_production} days')
    ax4.set_xlabel('Time (days)', fontweight='bold')
    ax4.set_ylabel('Inventory Level (units)', fontweight='bold')
    ax4.set_title('POQ Inventory Pattern Over One Cycle', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.ticklabel_format(style='plain', axis='y')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualizations
fig = create_poq_visualizations(params, results)
print("\nPOQ Analysis Dashboard created successfully!")

In [ ]:
# Additional validation: Compare with alternative integer solution
print(f"\n=== INTEGER SOLUTION VALIDATION ===")
print(f"Optimal Integer Solution: Q = {results['optimal_q_integer']:,} units")
print(f"Alternative Integer Solution: Q = {results['alternative_q']:,} units")
print(f"\nCost Comparison:")
print(f"Optimal Solution Cost: ${results['total_cost']:,.2f}")
print(f"Alternative Solution Cost: ${results['alternative_cost']:,.2f}")
print(f"Cost Difference: ${abs(results['total_cost'] - results['alternative_cost']):,.2f}")
print(f"Percentage Difference: {abs(results['total_cost'] - results['alternative_cost']) / results['total_cost'] * 100:.3f}%")

# Verify the mathematical relationships
print(f"\n=== MATHEMATICAL RELATIONSHIP VERIFICATION ===")
print(f"Production Rate (p): {params.p} units/day")
print(f"Demand Rate (d): {params.d} units/day")
print(f"Demand Rate Ratio (d/p): {results['demand_rate_ratio']:.3f}")
print(f"\nProduction Run Length: Q/p = {results['optimal_q_integer']:,} / {params.p} = {results['production_run_length_days']:.2f} days")
print(f"Consumption Cycle Length: Q/d = {results['optimal_q_integer']:,} / {params.d} = {results['consumption_cycle_length_days']:.2f} days")
print(f"Maximum Inventory: Q × (1 - d/p) = {results['optimal_q_integer']:,} × (1 - {results['demand_rate_ratio']:.3f}) = {results['max_inventory']:,.0f} units")

# Annual validation
annual_production_check = results['optimal_q_integer'] * results['num_production_runs_per_year']
print(f"\nAnnual Production Check:")
print(f"Optimal Q × Number of Runs = {results['optimal_q_integer']:,} × {results['num_production_runs_per_year']:.2f} = {annual_production_check:,.0f} units")
print(f"Annual Demand: {params.D:,} units")
print(f"Match Accuracy: {abs(annual_production_check - params.D) / params.D * 100:.2f}%")

In [ ]:
# What-if analysis: Different demand scenarios
def what_if_analysis(params: POQParameters) -> Dict:
    """
    Perform what-if analysis for different demand scenarios
    
    Arguments:
    params: Base POQParameters object
    
    Returns:
    Dictionary containing what-if analysis results
    """
    
    scenarios = [
        {'name': 'Low Demand', 'demand_multiplier': 0.7},
        {'name': 'Base Case', 'demand_multiplier': 1.0},
        {'name': 'High Demand', 'demand_multiplier': 1.3},
        {'name': 'Very High Demand', 'demand_multiplier': 1.5}
    ]
    
    scenario_results = []
    
    for scenario in scenarios:
        new_demand = params.D * scenario['demand_multiplier']
        new_daily_demand = params.d * scenario['demand_multiplier']
        
        # Ensure production rate still exceeds demand rate
        if new_daily_demand >= params.p:
            continue  # Skip invalid scenarios
        
        test_params = POQParameters(
            new_demand, params.S, params.H, params.p, new_daily_demand, params.working_days
        )
        
        _, q_opt, res = calculate_optimal_poq(test_params)
        
        scenario_results.append({
            'scenario': scenario['name'],
            'demand_multiplier': scenario['demand_multiplier'],
            'annual_demand': new_demand,
            'daily_demand': new_daily_demand,
            'optimal_q': q_opt,
            'total_cost': res['total_cost'],
            'num_runs': res['num_production_runs_per_year'],
            'max_inventory': res['max_inventory']
        })
    
    return scenario_results

# Perform what-if analysis
what_if_results = what_if_analysis(params)

print(f"\n=== WHAT-IF ANALYSIS: DEMAND SCENARIOS ===")
for result in what_if_results:
    print(f"\n{result['scenario']} (Demand Multiplier: {result['demand_multiplier']:.1f}):")
    print(f"  Annual Demand: {result['annual_demand']:,.0f} units")
    print(f"  Daily Demand: {result['daily_demand']:.0f} units/day")
    print(f"  Optimal Q: {result['optimal_q']:,} units")
    print(f"  Total Cost: ${result['total_cost']:,.0f}")
    print(f"  Production Runs: {result['num_runs']:.1f} per year")
    print(f"  Max Inventory: {result['max_inventory']:,.0f} units")

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical approach that provides:
- **Exact optimal solutions** for the POQ problem
- **Theoretical foundation** for understanding the cost trade-offs
- **Benchmark** for comparing heuristic and advanced algorithms
- **Analytical insights** into parameter sensitivities

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution for the mathematical model
- Provides clear understanding of cost relationships
- Fast computation for single-product problems
- Transparent and explainable methodology

**Cons:**
- Limited to simple, deterministic problems
- Cannot handle multiple products or complex constraints
- Assumes all parameters are known and constant
- May not scale well to complex real-world scenarios

### When to use this Tier
- **Single-product POQ problems** with stable demand
- **Educational purposes** to understand POQ fundamentals
- **Benchmarking** heuristic or metaheuristic solutions
- **Parameter sensitivity analysis** for strategic planning
- **Small-scale manufacturing** with simple production processes

## Summary

The mathematical formulation of the POQ problem provides the optimal production quantity of **8,944 tablets** per run with a total annual cost of **$10,731**. This solution balances setup costs ($5,365) and holding costs ($5,366) to minimize total costs while meeting the annual demand of 60,000 tablets.

**Key Insights:**
- The optimal batch size is significantly smaller than annual demand, requiring about 6.7 production runs per year
- Setup and holding costs are nearly equal at optimal solution (characteristic of EOQ-type problems)
- Maximum inventory reaches 5,366 units, about 60% of the production quantity
- Each production cycle lasts approximately 45 days (18 days production, 27 days consumption)

The mathematical approach serves as the foundation for more advanced solution methods that can handle multiple products, capacity constraints, and demand uncertainty.